In [32]:
# 0. Setup
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report


#Data Loading

In [33]:
PATH_COURSES = "https://raw.githubusercontent.com/Andrea-Pezzella/Course-Recommendation-System/refs/heads/main/data/courses.csv"
PATH_INFO    = "https://raw.githubusercontent.com/Andrea-Pezzella/Course-Recommendation-System/refs/heads/main/data/studentInfo(Starting).csv"

df_courses = pd.read_csv(PATH_COURSES)
df_info    = pd.read_csv(PATH_INFO)

print("courses shape:", df_courses.shape)
print("student Info shape:", df_info.shape)

df_courses.head()
df_info.head()


courses shape: (80, 4)
student Info shape: (32593, 12)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


#Data Preprocessing

In [34]:
courses = df_courses.copy()
df      = df_info.copy()

np.random.seed(0)   # makes randomness repeatable

# 1. Create 9000 users

n_users = 9000
user_ids = [f"user{idx:04d}" for idx in range(1, n_users + 1)]

# use replace=True so it never errors if df has < 9000 rows
base = df.sample(n=n_users, replace=True).reset_index(drop=True)
base["user_id"] = user_ids

fields = courses["field"].unique()
levels = ["Beginner", "Medium", "Senior"]

base["ufv_field"] = np.random.choice(fields, size=n_users)
base["ufv_level"] = np.random.choice(levels, size=n_users)
base["age"]       = np.random.randint(18, 36, size=n_users)

user_cols = [
    "user_id",
    "id_student",
    "gender",
    "age",
    "ufv_field",
    "ufv_level",
    "highest_education",
    "imd_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability",
]

user_profiles = base[user_cols].copy()

# 2. Assign users to all rows

df["user_id"] = np.random.choice(user_ids, size=len(df), replace=True)

# remove old versions of these columns (if they exist)
for col in user_cols:
    if col in df.columns and col != "user_id":
        df = df.drop(columns=col)

# add consistent user info to every row
df = df.merge(user_profiles, on="user_id", how="left")


# 3. Prepare courses and assign

courses = courses.copy()
courses["ufv_level"] = pd.cut(
    courses["course_number"],
    bins=[0, 199, 299, 999],
    labels=["Beginner", "Medium", "Senior"]
)

def assign_courses(group):
    field = group["ufv_field"].iloc[0]
    level = group["ufv_level"].iloc[0]

    sub = courses[(courses["field"] == field) & (courses["ufv_level"] == level)]

    # if no matching courses, just leave NaN and return
    if sub.empty:
        group["ufv_course_code"] = np.nan
        group["ufv_course_name"] = np.nan
        return group

    choices = sub[["course_code", "course_name"]].values
    idx = np.random.choice(len(choices), size=len(group), replace=True)

    group["ufv_course_code"] = choices[idx, 0]
    group["ufv_course_name"] = choices[idx, 1]
    return group

df = df.groupby("user_id", group_keys=False).apply(assign_courses)


# 4. Keep final columns and save

final_cols = [
    "user_id",
    "id_student",
    "code_module",
    "code_presentation",
    "final_result",
    "gender",
    "age",
    "ufv_field",
    "ufv_course_code",
    "ufv_course_name",
    "ufv_level",
    "highest_education",
    "imd_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability",
]

df_fixed = df[final_cols]


print("New studentInfo_fixed.csv saved")

New studentInfo_fixed.csv saved


/tmp/ipykernel_5959/2848315599.py:79: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("user_id", group_keys=False).apply(assign_courses)


In [35]:
PATH_STUDENT = "https://raw.githubusercontent.com/Andrea-Pezzella/Course-Recommendation-System/refs/heads/main/data/studentInfo_fixed.csv"

df_info = pd.read_csv(PATH_STUDENT)

print("studentInfo_fixed shape:", df_info.shape)

df_info.head()


studentInfo_fixed shape: (20547, 16)


,user_id,id_student,code_module,code_presentation,final_result,gender,age,ufv_field,ufv_course_code,ufv_course_name,ufv_level,highest_education,imd_band,num_of_prev_attempts,studied_credits,disability
0,user1538,531027,GGG,2014J,Pass,F,29,Media and Communications,media210,media210 - Media Ethics,Medium,A Level or Equivalent,0-10%,0,30,N
1,user4085,598914,DDD,2013B,Withdrawn,F,23,Media and Communications,media240,media240 - Podcasting,Medium,A Level or Equivalent,50-60%,1,180,N
2,user8569,264285,DDD,2014J,Fail,F,19,Computer science,comp285,comp285 - Cloud Computing,Medium,A Level or Equivalent,NaN,0,60,N
3,user2495,52797,BBB,2013J,Distinction,F,22,Creative Arts and Design,arts285,arts285 - Game Art,Medium,Lower Than A Level,30-40%,1,60,N
4,user2204,558146,FFF,2013B,Pass,F,19,Business and Management,bus260,bus260 - Project Management,Medium,A Level or Equivalent,80-90%,1,120,N


###Prepare target and features

In [36]:
df = df_info.copy()

# target: 1 if success, 0 otherwise
df["success"] = df["final_result"].isin(["Pass", "Distinction"]).astype(int)

# we will use these features (simple)
cat_features = [
    "gender",
    "ufv_level",
    "highest_education",
    "imd_band",
    "disability",
    "ufv_course_code"   # course itself as a category
]

num_features = [
    "age",
    "num_of_prev_attempts",
    "studied_credits"
]

# drop rows with missing key values
df = df.dropna(subset=cat_features + num_features + ["success"])

X = df[cat_features + num_features]
y = df["success"]

print("X shape:", X.shape)
print("y value counts:\n", y.value_counts())


X shape: (19695, 9)
y value counts:
 success
1    13829
0     5866
Name: count, dtype: int64


###Train and Test split

In [37]:
# Split the data into:
#  - X_train, y_train → used to train the model
#  - X_test,  y_test  → used to test the model on unseen data
# test_size=0.2 means 20% of the data is used for testing
# random_state=42 makes the split reproducible (same every time)
# stratify=y keeps the same success/fail ratio in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [38]:
# ColumnTransformer lets us apply different transformations to different groups of columns.

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),  # encode categorical features
        ("num", StandardScaler(), num_features),                        # scale numerical features
    ]
)

###Defining the models

Logistic Regression model

In [39]:
pipe_lr = Pipeline(steps=[
    ("preprocess", preprocess),                  # apply encoding + scaling
    ("model", LogisticRegression(max_iter=1000)) # the ML model
])


Random Forest model

In [40]:
pipe_rf = Pipeline(steps=[
    ("preprocess", preprocess),                  # same preprocessing steps
    ("model", RandomForestClassifier(
        n_estimators=200,     # number of trees
        random_state=42,      # reproducible results
        n_jobs=-1             # use all CPU cores for faster training
    ))
])


###Training & comparing

In [41]:
# Put both models into a dictionary so we can loop over them
models = {
    "LogisticRegression": pipe_lr,
    "RandomForest": pipe_rf
}

results = {}

# Train each model one by one
for name, model in models.items():
    print(f"\n Training {name} ")

    # Fit (train) the model on the training data
    model.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test)              # predicted labels (0 or 1)
    y_proba = model.predict_proba(X_test)[:, 1] # predicted probability of success

    # Accuracy = % of correct predictions
    acc = accuracy_score(y_test, y_pred)

    # ROC AUC = measures how well probabilities separate success vs fail
    auc = roc_auc_score(y_test, y_proba)

    results[name] = {"accuracy": acc, "roc_auc": auc}

    print(f"{name} accuracy:  {acc:.3f}")
    print(f"{name} ROC AUC:   {auc:.3f}")

print("\n Summary ")
for name, metrics in results.items():
    print(f"{name}: acc={metrics['accuracy']:.3f}, roc_auc={metrics['roc_auc']:.3f}")




 Training LogisticRegression 
LogisticRegression accuracy:  0.702
LogisticRegression ROC AUC:   0.505

 Training RandomForest 
RandomForest accuracy:  0.650
RandomForest ROC AUC:   0.508

 Summary 
LogisticRegression: acc=0.702, roc_auc=0.505
RandomForest: acc=0.650, roc_auc=0.508


###Choosing best model

In [42]:
# Variables to keep track of the best model found so far
best_name = None
best_auc = -1
best_model = None

# Loop through all model results
for name, metrics in results.items():

    # If this model has a higher AUC, update the best model info
    if metrics["roc_auc"] > best_auc:
        best_auc = metrics["roc_auc"]   # update best score
        best_name = name                # update best model name
        best_model = models[name]       # update best model itself

print(f"\nBest model (by ROC AUC): {best_name} with AUC = {best_auc:.3f}")



Best model (by ROC AUC): RandomForest with AUC = 0.508


###Course list for recommendation

In [43]:
course_table = df_courses[["course_code", "course_name", "field"]].drop_duplicates()
course_table = course_table.sort_values("course_code").reset_index(drop=True)

print("Number of unique courses:", len(course_table))
course_table.head()

Number of unique courses: 80


,course_code,course_name,field
0,arts105,arts105 - Design Fundamentals,Creative Arts and Design
1,arts115,arts115 - Drawing I,Creative Arts and Design
2,arts125,arts125 - Digital Media,Creative Arts and Design
3,arts135,arts135 - Typography,Creative Arts and Design
4,arts155,arts155 - Color Theory,Creative Arts and Design


###Interactive recommender

In [44]:
def collect_student_profile():
    print("Course Recommendation System (based on our dataset records)\n")

    while True:
        # Ask user for an ID
        sid_input = input(
            "Enter your student ID(e.g. 11391, 604204, 107028, 2083576): "
        ).strip()

        if sid_input == "":
            print("ID cannot be empty. Please try again.\n")
            continue

        # Look for this ID in either id_student
        matches = df[
            (df["id_student"].astype(str) == sid_input)
        ]

        if matches.empty:
            print("ID not found in the dataset. Please try again.\n")
            continue

        # Use the first matching row
        row = matches.iloc[0]

        # Show basic info about this student
        print("\nStudent found. Using this profile:")
        print(row[[
            "user_id",
            "id_student",
            "gender",
            "age",
            "ufv_field",
            "ufv_level",
            "highest_education",
            "imd_band",
            "num_of_prev_attempts",
            "studied_credits",
            "disability"
        ]])

        # Build the profile used by the model
        profile = {
            "gender": row["gender"],
            "age": row["age"],
            "ufv_level": row["ufv_level"],
            "highest_education": row["highest_education"],
            "imd_band": row["imd_band"],
            "disability": row["disability"],
            "num_of_prev_attempts": row["num_of_prev_attempts"],
            "studied_credits": row["studied_credits"],
        }

        # Student field (used to prefer same-field courses)
        student_field = row["ufv_field"]

        return profile, student_field


def recommend_courses(best_model, top_k=5):
    # 1) Get profile + field from dataset
    profile, student_field = collect_student_profile()

    # 2) Build a new DataFrame with one row per course
    rows = []
    for _, row in course_table.iterrows():
        d = dict(profile)                     # copy student info
        d["ufv_course_code"] = row["course_code"]  # add course code
        rows.append(d)

    df_new = pd.DataFrame(rows, columns=cat_features + num_features)

    # 3) Predict success probabilities for all courses
    probs = best_model.predict_proba(df_new)[:, 1]

    # 4) Attach probabilities to the course table
    rec_df = course_table.copy()
    rec_df["pred_success_prob"] = probs

    # 5) First pick courses in the same field
    same_field = rec_df[rec_df["field"] == student_field] \
        .sort_values("pred_success_prob", ascending=False)

    # If enough courses in the same field, use only them
    if len(same_field) >= top_k:
        final_df = same_field.head(top_k)
    else:
        # Otherwise, fill with best courses from other fields
        remaining = top_k - len(same_field)
        others = rec_df[rec_df["field"] != student_field] \
            .sort_values("pred_success_prob", ascending=False) \
            .head(remaining)
        final_df = pd.concat([same_field, others])

    # 6) Print final recommended list
    print(f"\n---> Student field: {student_field}")
    print("\n---> Top recommended courses for this student:\n")
    for _, r in final_df.iterrows():
        print(
            f"{r['course_code']}: {r['course_name']} "
            f"(Field: {r['field']}, predicted success prob: {r['pred_success_prob']:.2f})"
        )

    return final_df


###Input:

In [46]:

_ = recommend_courses(best_model, top_k=5)


Course Recommendation System (based on our dataset records)

Enter your student ID(e.g. 11391, 604204, 107028, 2083576): 11391

Student found. Using this profile:
user_id                            user0001
id_student                            11391
gender                                    F
age                                      19
ufv_field               History and Culture
ufv_level                          Beginner
highest_education          HE Qualification
imd_band                            90-100%
num_of_prev_attempts                      0
studied_credits                         240
disability                                N
Name: 9495, dtype: object

---> Student field: History and Culture

---> Top recommended courses for this student:

hist125: hist125 - Cultural Studies (Field: History and Culture, predicted success prob: 0.92)
hist135: hist135 - Modern Europe (Field: History and Culture, predicted success prob: 0.85)
hist105: hist105 - World History (Field: History a